# Setup 2 — carga histórica (bootstrap da base)

**Objetivo:** puxar histórico de cada fonte no intervalo **`INICIO` .. `FIM`** que você define
na primeira célula. O boletim e o cálculo rodam para **todos os dias úteis do intervalo** (não só
as pontas). As fontes de indicativa/curva só entregam uma janela recente — o notebook pede o
máximo que cada uma guarda.

**Rode depois do `setup_1_teste.ipynb`** (que já provou cada fluxo). **Pode dar `Run All`** —
nada aborta; cada bloco mostra `[OK]`/`[FALHA]` e a célula final confere as contagens. Se um
bloco falhar, corrija e **re-rode só ele** (todos idempotentes; NTN-B/cálculo pulam o já feito).

**Não apaga dados.** Tudo é `CREATE TABLE IF NOT EXISTS` + **UPSERT** — rodar de novo por cima de
uma base que já tem dados é seguro e apenas adiciona/atualiza. Pode puxar este código num PC que
já tenha `trades.db` populado.

**Anbima Data (o passo mais pesado):** se a `InfoAtivos` **já estiver populada**, o notebook roda
só o **incremental** do período (novos tickers) e **pula o scraping do universo completo**. Base
vazia → roda completo. Force o completo com `FORCAR_ANBIMA_FULL = True` naquele bloco.

⚠️ **É demorado** (boletim dia a dia + cálculo dia a dia). Dá pra `Run All` e voltar depois.
**Rode a partir da pasta `code/`.**

## Config — janela alvo e limites reais de cada fonte

In [ ]:
import sys
from pathlib import Path
from datetime import date, timedelta

if not (Path.cwd() / "scripts").exists():
    raise SystemExit(f"Rode a partir da pasta code/. cwd atual: {Path.cwd()}")
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "scripts"))
import pipeline_core as pc
from lib.db import get_db

get_db().close()   # cria/verifica o schema — NAO apaga dados (tudo IF NOT EXISTS + UPSERT)

# ================== JANELA DA CARGA — EDITE AQUI ==================
# INICIO e FIM definem o intervalo COMPLETO (inclusive). O boletim e o calculo
# rodam para TODOS os dias uteis entre os dois — nao so as pontas.
INICIO = date(2026, 6, 1)     # 1o dia do intervalo
FIM    = date(2026, 7, 1)     # ultimo dia do intervalo
# =================================================================

HOJE = date.today()
# Limites REAIS de cada fonte (nao adianta pedir alem do que ela guarda):
INI_IND     = max(INICIO, HOJE - timedelta(days=125))  # deb/NTN-B: fonte guarda ~4 meses
FIM_IND     = min(FIM, HOJE)                            # indicativas nao existem no futuro
DIAS_DI     = 20    # curva DI B3: ~20 pregoes (so os mais recentes existem)
DIAS_CRICRA = 5     # CRI/CRA: portal ~5 pregoes

DIAS = [d.isoformat() for d in pc.dias_uteis_entre(INICIO, FIM)]  # TODOS os dias uteis do intervalo
assert DIAS, "Intervalo sem dias uteis — confira INICIO/FIM."
print(f"Janela: {INICIO} .. {FIM}  ->  {len(DIAS)} pregoes")
print(f"  1o: {DIAS[0]}  ...  ultimo: {DIAS[-1]}")
print(f"Indicativas (deb/NTN-B): {INI_IND} .. {FIM_IND}")

## Scraping — 1 bloco por fonte
Cada fonte na janela máxima que entrega. **Anbima Data (o mais pesado) por último.**

In [ ]:
# 1. Boletim B3 (negocios) — dia a dia sobre TODO o intervalo INICIO..FIM.
#    (o scraper agora baixa 1 pregao por vez internamente; nao pula os dias do meio)
pc.boletim(INICIO.isoformat(), FIM.isoformat())

In [ ]:
# 2. FI Analytics planilha (caracteristicas) — snapshot atual (Playwright+login)
pc.fianalytics()

In [ ]:
# 3. Anbima debentures (indicativas) — ~4 meses (fonte skipa 404 fora da janela)
pc.anbima_deb(INI_IND.isoformat(), FIM_IND.isoformat())

In [ ]:
# 4. Anbima NTN-B (MtM) — ~4 meses. Duration paralela (4 workers) + skip do ja feito.
pc.ntnb(INI_IND.isoformat(), FIM_IND.isoformat())

In [ ]:
# 5. Curva DI B3 (MtM) — ultimos ~20 pregoes ate FIM (limite da fonte)
for d in pc.ultimos_n_dias_uteis(DIAS_DI, ref=FIM_IND):
    pc.curva_di(d)

In [ ]:
# 6. Anbima CRI/CRA (indicativas, Playwright) — ultimos ~5 pregoes ate FIM (limite do portal)
for d in pc.ultimos_n_dias_uteis(DIAS_CRICRA, ref=FIM_IND):
    pc.anbima_cricra(d)

In [ ]:
# 7. Outstanding via Bloomberg — SO NO BANCO (no PC pessoal da [FALHA], tudo bem)
pc.outstanding(INICIO.isoformat(), FIM.isoformat())

In [ ]:
# 8. Anbima Data (caracteristicas + fluxo) — o passo MAIS PESADO.
#    Base VAZIA        -> universo COMPLETO (--mode full).
#    Base JA POPULADA  -> so INCREMENTAL do periodo (novos tickers); NAO re-raspa o universo.
#    Roda antes do calculo (match/spread usam InfoAtivos que este passo preenche).
FORCAR_ANBIMA_FULL = False   # True forca o --mode full mesmo com a base ja populada

import sqlite3 as _sq
_n_info = _sq.connect('data/trades.db').execute("SELECT COUNT(*) FROM InfoAtivos").fetchone()[0]
if FORCAR_ANBIMA_FULL or _n_info == 0:
    print(f"InfoAtivos = {_n_info:,} -> Anbima Data COMPLETO (--mode full)")
    pc.anbima_data(full=True)
else:
    print(f"InfoAtivos = {_n_info:,} ja populado -> Anbima Data INCREMENTAL {INICIO}..{FIM} (pula universo completo)")
    pc.anbima_data(INICIO.isoformat(), FIM.isoformat())

## Cálculo — percorre todos os pregões da janela (`DIAS`)
Idempotente: re-rodar pula o já feito. Ordem: taxa → filtrar → spread Anbima → match → spread over → relatório.

In [ ]:
# 9. Taxa por trade (cascata FI Analytics -> B3) — ja paralelo + cache interno
for X in DIAS:
    pc.calc_taxa(X)

In [ ]:
# 10. Filtrar (VALIDO / FUNDO / BROKER / PF)
for X in DIAS:
    pc.filtrar(X)

In [ ]:
# 11. Spread Anbima das indicativas
for X in DIAS:
    pc.spread_anbima(X)

In [ ]:
# 12. Match de referencia (global, sem data)
pc.match_ref()

In [ ]:
# 13. Spread over dos trades
for X in DIAS:
    pc.spread_over(X)

In [ ]:
# 14. Gerar relatorio (toda a base)
pc.relatorio()

## Conferência — a base foi montada?
Contagens > 0 nas tabelas principais = deu certo. Se algo ficou zerado, veja qual bloco deu `[FALHA]` e re-rode só ele (+ os de cálculo).

In [ ]:
import sqlite3
c = sqlite3.connect('data/trades.db')
for t in ['NegociosBrutos','NegociosProcessados','InfoAtivos','AnbimaIndicativos','MtmAnbima','FluxoAtivos','Outstanding']:
    print(f'  {t:22s}: {c.execute("SELECT COUNT(*) FROM "+t).fetchone()[0]:>9,} linhas')
c.close()